# FRIDAY DesktopIntent-0.5B — v10 Definitive 🖥️⚡

Trains the **`friday:v10-definitive`** router from the clean **Canonical Master Dataset (`v10_blend`)** on Hugging Face (`ThisisDarkstar/friday-desktop-intent`).

### 🌟 What Makes v10 Definitive:
1. **Zero Multi-Version Schema Conflicts**: 100% compliant with live FRIDAY MCP tool signatures (`app_name`, `title_keyword`, `confirmed: true`).
2. **GUI Single-Key Safety**: Single keys (`enter`, `tab`, `esc`) strictly map to `press_key(key=...)`, eliminating `UNKNOWN_SHORTCUT` crashes.
3. **Read-Only vs Mutation Guard**: Questions (*"Is dark mode on?"*) map to `get_*` introspection tools, avoiding state mutations.
4. **Zero Window Hallucinations**: Standardized window maximization without phantom tools like `full_screen_window`.
5. **Planned Tools Preserved**: All 18 future tools (`calculate`, `take_note`, `obs_record`, `minimize_all_windows`, etc.) are intact.
6. **Strict Deduplication**: 7,632 duplicate prompts pruned; sharp logit distributions ensure sub-200ms latency.

### ⚙️ Training Specifications:
- **Base Model**: `Qwen/Qwen2.5-0.5B-Instruct` (Fresh Base)
- **Dataset**: `v10_blend` (29,551 train / 3,196 validation)
- **LoRA Config**: Rank 32, Alpha 64, LR 7e-5, Cosine Scheduler, 2 Epochs
- **Output Artifact**: `friday-v10-definitive-q8.gguf` (~531 MB)

## 1. Install Dependencies & Clone llama.cpp

In [ ]:
!pip install -q -U transformers peft datasets accelerate trl huggingface_hub gguf sentencepiece protobuf torchao
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git

## 2. Dataset Source Verification (Hugging Face)

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download
import json

HF_DATASET_ID = "ThisisDarkstar/friday-desktop-intent"
print(f"[*] Loading config 'v10_blend' from: {HF_DATASET_ID}")

report_path = hf_hub_download(HF_DATASET_ID, "subsets/v10_blend/blend_report.json",
                              repo_type="dataset")
report = json.loads(Path(report_path).read_text(encoding="utf-8"))
print(f"[+] Train exposures : {report['train_total']:,}")
print(f"[+] Validation rows : {report['validation_total']:,}")
print(f"[+] Exposure shares : {report['exposure_share_pct']}")

## 3. GPU Environment Check

In [ ]:
import torch
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available  : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 4. Load the v10 Definitive Blend

In [ ]:
from datasets import load_dataset

try:
    v10 = load_dataset(HF_DATASET_ID, "v10_blend")
except Exception as e:
    print(f"[*] Standard config load failed ({e}); using direct JSON URLs fallback")
    v10 = load_dataset("json", data_files={
        "train": f"https://huggingface.co/datasets/{HF_DATASET_ID}/resolve/main/subsets/v10_blend/train.jsonl",
        "validation": f"https://huggingface.co/datasets/{HF_DATASET_ID}/resolve/main/subsets/v10_blend/validation.jsonl",
    })

train_ds, val_ds = v10["train"], v10["validation"]
print(f"[+] Train dataset      : {len(train_ds):,} samples")
print(f"[+] Validation dataset : {len(val_ds):,} samples")
print(f"[+] Example prompt     : {train_ds[0]['messages'][1]['content']}")
print(f"[+] Example target     : {train_ds[0]['messages'][2]['content']}")

## 5. Base Model Setup & LoRA Initialization

In [ ]:
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

BASE_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "./results_0.5b_v10_definitive"
MERGED_DIR = "./friday-0.5b-v10-definitive-merged"

print(f"[*] Initializing fresh base model: {BASE_MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map={"": 0},
    trust_remote_code=True,
)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.config.use_cache = False

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 6. Prompt Masking & Tokenization

In [ ]:
MAX_LENGTH = 512

def format_chatml(batch, max_length=MAX_LENGTH):
    input_ids_list, labels_list = [], []
    for messages in batch["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        prompt_text = tokenizer.apply_chat_template(
            messages[:-1], tokenize=False, add_generation_prompt=True)
        full = tokenizer(text, truncation=True, max_length=max_length)["input_ids"]
        prompt_len = len(tokenizer(prompt_text, truncation=True, max_length=max_length)["input_ids"])
        labels = [-100] * prompt_len + full[prompt_len:]
        labels = (labels + [-100] * len(full))[:len(full)]
        input_ids_list.append(full)
        labels_list.append(labels)
    return {"input_ids": input_ids_list, "labels": labels_list}

print("[*] Tokenizing datasets with ChatML masking...")
tokenized_train = train_ds.map(format_chatml, batched=True, remove_columns=train_ds.column_names)
tokenized_val = val_ds.map(format_chatml, batched=True, remove_columns=val_ds.column_names)
print(f"[+] Tokenized {len(tokenized_train):,} train / {len(tokenized_val):,} val")

## 7. Fine-Tuning Execution (2 Epochs, Cosine Decay)

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq, TrainerCallback

class RealtimeLineLogger(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            pct = (state.global_step / state.max_steps) * 100 if state.max_steps else 0.0
            print(f"[Step {state.global_step:4d}/{state.max_steps}] ({pct:5.1f}%) "
                  f"loss={logs['loss']:.4f} lr={logs.get('learning_rate', 0):.2e} "
                  f"epoch={logs.get('epoch', state.epoch):.2f}", flush=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=7e-5,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    num_train_epochs=2,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=25,
    disable_tqdm=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    optim="adamw_torch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer, pad_to_multiple_of=8, return_tensors="pt"),
    callbacks=[RealtimeLineLogger()],
)

print("[*] Starting training for friday:v10-definitive...")
trainer.train()
print("[+] Training completed successfully!")

## 8. Merge LoRA Weights into Standalone Model

In [ ]:
import os
model.config.use_cache = True
print("[*] Merging LoRA adapters with base model...")
merged_model = model.merge_and_unload()
os.makedirs(MERGED_DIR, exist_ok=True)
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print(f"[+] Saved standalone merged model to: {MERGED_DIR}")

## 9. Export Q8_0 GGUF Quantization

In [ ]:
print("[*] Converting model to GGUF Q8_0 format...")
!python llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outtype q8_0 --outfile ./friday-v10-definitive-q8.gguf
!ls -lh *.gguf

# Direct Colab Download Helper
try:
    from google.colab import files
    print("[*] Triggering direct browser download in Colab...")
    files.download("./friday-v10-definitive-q8.gguf")
except Exception:
    print("[!] Running outside Colab or headless environment. Download file from file browser.")

## 10. Verification & Local Ollama Deployment

After downloading `friday-v10-definitive-q8.gguf`, register it into your local Ollama:

```powershell
# Create Modelfile
Set-Content -Path Modelfile -Value @"
FROM ./friday-v10-definitive-q8.gguf
TEMPLATE """<|im_start|>system
{{ .System }}<|im_end|>
<|im_start|>user
{{ .Prompt }}<|im_end|>
<|im_start|>assistant
"""
PARAMETER stop "<|im_start|>"
PARAMETER stop "<|im_end|>"
PARAMETER temperature 0.0
"""@

# Register in Ollama
ollama create friday:v10-definitive -f Modelfile
ollama run friday:v10-definitive "type 'test message' and press enter"
```